# Week 19, Day 6: Forge: Week 19 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w019/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import lgamma, log, exp
print("1. Beta(1 + 3, 1 + 1) = Beta(4, 2)")
print("2. (2 + 10) / (1 + 2) =", (2 + 10) / (1 + 2))
print("3. 1 / (n + 1) =", 1 / 10)

### Exercise: Part A: a Bayes factor for tuning

Write `log_mean_exp(a)` and `log_evidence(k, X, n_draws, rng)` (draw weight vectors from N(0, 1), compute each one's Poisson log-likelihood, and return the log of their mean likelihood). For a tuned and an untuned neuron, 40 trials each, compute the log Bayes factor of M1 against M0 with 100,000 prior draws.

In [ ]:
rng = np.random.default_rng(2011)
def log_mean_exp(a):
    mx = a.max()
    return ...

def log_evidence(k, X, n_draws, rng):
    W = rng.normal(size=(n_draws, X.shape[1]))
    eta = W @ X.T                                          # draws x trials
    ll = ...                                               # Poisson log-likelihood of each draw (drop log k!)
    return log_mean_exp(ll)

theta = rng.uniform(0, np.pi, 40)
X1 = np.column_stack([np.ones(40), np.cos(2 * theta), np.sin(2 * theta)])
X0 = X1[:, :1]
neurons = {"tuned": rng.poisson(np.exp(1.2 + 0.7 * np.cos(2 * (theta - 1.0)))), "untuned": rng.poisson(np.exp(1.2) * np.ones(40))}
log_bf = {name: log_evidence(k, X1, 100000, rng) - log_evidence(k, X0, 100000, rng) for name, k in neurons.items()}
print({name: round(v, 2) for name, v in log_bf.items()}, " (log Bayes factors, tuned against constant)")

In [ ]:
import numpy as np
def test_lse():
    "log_mean_exp is a stable log of the mean of exponentials"
    assert abs(log_mean_exp(np.array([1000.0, 1000.0])) - 1000.0) < 1e-9

def test_tuned():
    "for the tuned neuron the Bayes factor favours tuning by more than 100 to 1"
    assert log_bf["tuned"] > np.log(100)

def test_untuned():
    "for the untuned neuron it favours the constant model (log Bayes factor below 0)"
    assert log_bf["untuned"] < 0

test_lse()
test_tuned()
test_untuned()
print("All checks passed")

### Exercise: Part B: forgetting helps in a volatile world

Write `track(outcomes, decay)` that starts from a = b = 1 and, before each trial, sets a = 1 + decay (a − 1) and b = 1 + decay (b − 1), then adds the outcome, recording the posterior mean before each update. Compare decay 1 (no forgetting) and 0.9 in a static world and a volatile one (400 trials each), storing mean squared errors against the true probability in `err[world][learner]`.

In [ ]:
def track(outcomes, decay):
    a = b = 1.0
    means = []
    for o in outcomes:
        a, b = 1 + decay * (a - 1), 1 + decay * (b - 1)
        means.append(...)
        a, b = ..., ...
    return np.array(means)

r_b = np.random.default_rng(2007)
T = 400
worlds = {"static": np.full(T, 0.7), "volatile": np.where((np.arange(T) // 40) % 2 == 0, 0.2, 0.8)}
err = {}
for name, p_true in worlds.items():
    outcomes = (r_b.random(T) < p_true).astype(int)
    err[name] = {lab: np.mean((track(outcomes, d) - p_true) ** 2) for lab, d in [("full", 1.0), ("forget", 0.9)]}
print({w: {k: round(v, 4) for k, v in e.items()} for w, e in err.items()})

In [ ]:
import numpy as np
def test_static():
    "in a static world (p fixed at 0.7), the non-forgetting learner has the smaller error"
    assert err["static"]["full"] < err["static"]["forget"]

def test_volatile():
    "in the volatile world, the forgetting learner has at least 30% less error"
    assert err["volatile"]["forget"] < 0.7 * err["volatile"]["full"]

test_static()
test_volatile()
print("All checks passed")

### Exercise: Part C: coverage of credible intervals

For each prior, compute each synapse's posterior Beta(a + k, b + 10 − k), its 95% equal-tailed interval from 2,000 posterior samples, and the fraction of synapses whose true probability lies inside. Store the fractions in `cover['correct']` and `cover['wrong']`.

In [ ]:
r_c = np.random.default_rng(19)
p_syn = r_c.beta(2, 5, 2000)
k_syn = r_c.binomial(10, p_syn)
cover = {}
for name, (a0, b0) in {"correct": (2, 5), "wrong": (40, 40)}.items():
    samples = r_c.beta(a0 + k_syn[:, None], b0 + 10 - k_syn[:, None], (2000, 2000))   # synapses x samples
    lo, hi = ...
    cover[name] = ...
print({k: round(v, 3) for k, v in cover.items()})

In [ ]:
import numpy as np
def test_correct():
    "with the correct prior, the intervals contain the true probability 93% to 97% of the time"
    assert 0.93 < cover["correct"] < 0.97

def test_wrong():
    "with the overconfident wrong prior, coverage falls below 0.5"
    assert cover["wrong"] < 0.5

test_correct()
test_wrong()
print("All checks passed")